Save as: module5-simulation-lab.ipynb

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncap/micro-course/blob/main/module-05/module5-simulation-lab.ipynb)

# Lab 5 — Nudging Silicon Populations
### Default effects in behavioral vs. rational LLM agents
*Microeconomic Theory for Experimentalists & Applied Microeconomists*

**Research question:** Do default effects in retirement-plan enrollment
emerge in LLM agent populations, and — the sharp test — do they **interact
with the agent's discounting persona** the way theory says they should?

**Theory predictions** (state these as hypotheses *before* running; see the
student handout):

- **P1.** Exponential agents with trivial switching costs: default effect ≈ 0.
- **P2.** Naive present-biased agents: large default effect (procrastination
  on the switching action, whichever direction the default points).
- **P3.** Sophisticated present-biased agents: intermediate, and more likely
  to enroll under opt-in than naifs (they anticipate their own inertia).

**Human benchmark:** Madrian & Shea (2001, *QJE*). Automatic enrollment
raised 401(k) participation of new hires from ~37% to ~86% — a ~49 pp
default effect. Contribution rates clustered hard at the 3% default.

**How to run:** no prior Python experience needed. Run the cells top to
bottom. Your required modification (handout Section 3) goes **only** in the
cells marked **CHANGE HERE**. Course conventions: fixed reported
temperature, persona grid, parse-failure logging, prompt logs saved for
your write-up appendix, robustness section at the end.


## Setup

The first cell installs dependencies (Colab). The second asks for your
Anthropic API key via a hidden prompt — the key is never stored in the
notebook file.


In [ ]:
%pip install anthropic pandas matplotlib --quiet

In [ ]:
import itertools
import json
import re
import time
from getpass import getpass

import pandas as pd

import anthropic

# Never hard-code API keys. getpass keeps the key out of the notebook file.
API_KEY = getpass("Paste your Anthropic API key: ")
client = anthropic.Anthropic(api_key=API_KEY)

MODEL = "claude-sonnet-5"   # swap in the robustness cell
TEMPERATURE = 1.0            # fixed and reported — part of the design
N_PER_CELL = 30              # decisions per persona x treatment cell

## Experimental design — CHANGE HERE (all three blocks live in this cell)

3 discounting personas × 2 defaults × 2 incomes = 12 cells × 30 = 360 calls.

The discounting personas are the treatment of theoretical interest. Note the
design choice: personas are described in **plain behavioral language**, not
with Greek letters. If we wrote "you have beta = 0.7", the model would simply
compute the beta–delta answer — that tests arithmetic, not behavior.
Describing dispositions and letting behavior emerge is the closer analogue to
a human subject pool. (PS5 Part C asks what happens when you violate this and
parameterize explicitly — try it in the robustness section.)

Your required modification will usually mean **(1)** adding a factor to
`PERSONA_GRID` (e.g. `"reminder": ["no", "yes"]` or
`"default_source": ["unstated", "random"]`), **(2)** weaving it into
`build_prompt` *without changing any other wording across conditions*, and
**(3)** touching `parse_response` only if your answer format changes.


In [ ]:
# 3 discounting personas x 2 defaults x 2 incomes = 12 cells x 30 = 360 calls.
#
# The discounting personas are the treatment of theoretical interest. Note the
# design choice: we describe the persona in PLAIN BEHAVIORAL LANGUAGE, not
# with the Greek letters. If we wrote "you have beta = 0.7", the model would
# simply compute the beta-delta answer — that tests arithmetic, not behavior.
# Describing dispositions and letting behavior emerge is the closer analogue
# to a human subject pool. (PS5 Part C asks what happens when you violate
# this and parameterize explicitly — try it in the robustness cell.)
#
# YOUR REQUIRED MODIFICATION (pick one from handout Section 3) will usually
# mean adding a factor to PERSONA_GRID here (e.g., "reminder": ["no", "yes"]
# or "default_source": ["unstated", "random"]) and using it in build_prompt.

PERSONAS = {
    "exponential": (
        "You are financially disciplined. When you decide something is "
        "worth doing, you do it immediately; paperwork never sits on your "
        "desk. You weigh future benefits consistently, whether they arrive "
        "next month or in thirty years."
    ),
    "naive_present_biased": (
        "You strongly prefer enjoying money now over locking it away, and "
        "you tend to put off administrative tasks, telling yourself you'll "
        "handle them next week. You are confident that next week you will "
        "actually do it."
    ),
    "sophisticated_present_biased": (
        "You strongly prefer enjoying money now over locking it away, and "
        "you tend to put off administrative tasks. You KNOW this about "
        "yourself: you know that if you postpone something, you will likely "
        "keep postponing it, so you sometimes act now precisely to protect "
        "yourself from your own future procrastination."
    ),
}

PERSONA_GRID = {
    "discounting": list(PERSONAS),
    "default": ["opt_in", "opt_out"],            # the nudge treatment
    "income": ["$38,000 per year", "$95,000 per year"],
}

DEFAULT_CONTRIBUTION = 3   # percent of salary — Madrian & Shea's default rate
MAX_CONTRIBUTION = 15


def build_prompt(persona: dict) -> str:          # CHANGE HERE (2 of 3)
    """Decision prompt: new-hire retirement plan enrollment.

    Design rules (they matter for validity):
    - Persona first, situation second, response format last.
    - Neutral language (no 'save more!', no hint that enrolling is 'correct').
    - Switching costs described identically and as trivially small in both
      default conditions, so the rational benchmark for the default effect
      is ~0 (this is the identifying assumption — see handout).
    If your modification adds a treatment factor, weave it into the text
    below WITHOUT changing any other wording across conditions.
    """
    if persona["default"] == "opt_in":
        default_text = (
            "By default you are NOT enrolled in the plan. If you want to "
            "enroll, you must submit a one-page online form; it takes about "
            "ten minutes, and you can do it any time."
        )
    else:  # opt_out
        default_text = (
            f"By default you ARE enrolled in the plan at a "
            f"{DEFAULT_CONTRIBUTION}% contribution rate. If you want to "
            "leave the plan or change the rate, you must submit a one-page "
            "online form; it takes about ten minutes, and you can do it "
            "any time."
        )

    return (
        f"{PERSONAS[persona['discounting']]}\n\n"
        f"You just started a new job paying {persona['income']}. The employer "
        "offers a retirement savings plan: you may contribute between 1% and "
        f"{MAX_CONTRIBUTION}% of salary, the employer matches contributions "
        "up to 3%, and contributions are locked until retirement.\n\n"
        f"{default_text}\n\n"
        "It is the end of your first week. Decide what you actually do this "
        "week (not what you intend to do eventually).\n\n"
        'Respond with ONLY a JSON object: {"enrolled": true or false, '
        '"contribution_pct": <integer 0-15>} describing your plan status at '
        "the end of this week. If not enrolled, contribution_pct is 0."
    )


def parse_response(text: str) -> dict | None:    # CHANGE HERE (3 of 3)
    """Extract the decision. Return None on failure — never guess a value.

    Only edit this if your modification changes the answer format — and keep
    the same pattern: match, range-check, return None when anything is off.
    """
    match = re.search(
        r'\{\s*"enrolled"\s*:\s*(true|false)\s*,\s*"contribution_pct"\s*:\s*(\d+)\s*\}',
        text, re.IGNORECASE,
    )
    if match:
        enrolled = match.group(1).lower() == "true"
        pct = int(match.group(2))
        if 0 <= pct <= MAX_CONTRIBUTION and (enrolled or pct == 0):
            return {"enrolled": enrolled, "contribution_pct": pct}
    return None

## Run the experiment *(do not modify)*

Runs every persona × treatment cell, logs parse failures instead of dropping
them, saves `lab5_results.csv` and `lab5_prompts_log.json` (required in your
write-up appendix). ~360 API calls: expect a few minutes and a small API
spend.


In [ ]:
def run_experiment() -> pd.DataFrame:
    keys = list(PERSONA_GRID)
    cells = [dict(zip(keys, combo)) for combo in itertools.product(*PERSONA_GRID.values())]
    print(f"{len(cells)} cells x {N_PER_CELL} obs = {len(cells) * N_PER_CELL} decisions")

    rows = []
    for cell in cells:
        prompt = build_prompt(cell)
        for i in range(N_PER_CELL):
            try:
                response = client.messages.create(
                    model=MODEL,
                    max_tokens=150,
                    temperature=TEMPERATURE,
                    messages=[{"role": "user", "content": prompt}],
                )
                raw = response.content[0].text
                decision = parse_response(raw)
            except Exception as err:            # log failures; never drop silently
                raw, decision = f"ERROR: {err}", None
                time.sleep(5)
            rows.append({
                **cell, "rep": i,
                "enrolled": None if decision is None else decision["enrolled"],
                "contribution_pct": None if decision is None else decision["contribution_pct"],
                "raw": raw, "model": MODEL, "temperature": TEMPERATURE,
            })
        done = [r for r in rows if r["enrolled"] is not None]
        print(f"  cell {cell} done ({len(done)}/{len(rows)} parsed)")

    df = pd.DataFrame(rows)
    df.to_csv("lab5_results.csv", index=False)
    # Save exact prompts — required in the write-up appendix.
    with open("lab5_prompts_log.json", "w") as f:
        json.dump({str(c): build_prompt(c) for c in cells}, f, indent=2)
    return df


df = run_experiment()

## Analysis vs. human benchmark *(do not modify)*

Compares silicon enrollment rates to Madrian & Shea (2001): ~37% opt-in vs.
~86% opt-out, and contribution clustering at the 3% default. The
persona-by-default table is the H2 interaction test.


In [ ]:
# Madrian & Shea (2001): enrollment ~37% under opt-in, ~86% under opt-out
# (new hires); contributions pile up at the 3% default under auto-enrollment.
HUMAN_OPTIN, HUMAN_OPTOUT = 0.37, 0.86

valid = df.dropna(subset=["enrolled"]).copy()
valid["enrolled"] = valid["enrolled"].astype(bool)

print(f"\nParse-failure rate: {df['enrolled'].isna().mean():.3f} "
      "(report this in Section 4 of the write-up)")

print("\n=== Enrollment rate by persona x default ===")
table = (valid.groupby(["discounting", "default"])["enrolled"]
         .agg(["mean", "count"]).round(3))
print(table)

print("\n=== Default effect (opt_out minus opt_in), by persona ===")
rates = valid.groupby(["discounting", "default"])["enrolled"].mean().unstack()
rates["default_effect_pp"] = ((rates["opt_out"] - rates["opt_in"]) * 100).round(1)
print(rates.round(3))
print("\nTheory: effect ~0 for 'exponential', largest for 'naive'.")
print(f"Human benchmark (pooled population): "
      f"{(HUMAN_OPTOUT - HUMAN_OPTIN) * 100:.0f} pp")

print("\n=== Contribution anchoring at the default (opt_out enrollees) ===")
anchored = valid[(valid["default"] == "opt_out") & valid["enrolled"]]
share_at_default = (anchored["contribution_pct"] == DEFAULT_CONTRIBUTION).mean()
print(f"Share contributing exactly {DEFAULT_CONTRIBUTION}%: {share_at_default:.2f} "
      "(Madrian & Shea: strong clustering at the default)")

## Robustness — required in every lab

Rerun at least the 'exponential x opt_out' and 'naive x opt_in' cells with:
  (a) a paraphrased prompt (rewrite build_prompt's wording, same content);
  (b) a second model (change MODEL in Cell 2);
  (c) the EXPLICIT parameterization ("your beta is 0.7...") — does behavior
      snap to the textbook prediction? What does that tell you about what
      the persona conditioning is actually doing?
Report in Section 4 of the write-up whether the persona x default
interaction survives (a)-(c). If your interaction dies under paraphrase, it
is a fact about your prompt, not about silicon behavioral agents.

LIMITS OF SILICON SUBJECTS: these agents have read the literature you are
benchmarking against; they face no real stakes; their "types" are prompt
text, not preferences. A match with Madrian & Shea is not validation and a
divergence is not failure — in both cases the deliverable is the mechanism
and the design change it implies for the human study (write-up Section 6).